# Notebook 07: Uncertainty Estimation

Wraps the best-performing transformer with an **aleatoric uncertainty head**
and evaluates whether uncertainty scores are clinically meaningful:
do high-uncertainty predictions correspond to prediction errors?

Cells run in order; relies on trained checkpoints from notebooks 04 / 04b.

In [ ]:
import sys, os, json, warnings
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings('ignore')

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

from src.utils.config               import CFG
from src.models.uncertainty_head    import AleatoricWrapper, aleatoric_loss
from src.training.train_uncertainty import run_uncertainty_experiment
from src.inference.pipeline         import ECGInferencePipeline
from src.inference.loaders          import load_hubert_blocks, load_hubert_peft, load_leadwise
from src.preprocessing.label_utils  import load_all_labels
from src.preprocessing.dataset_full import ECGDatasetFull

DATA_PATH    = CFG['data']['path']
RESULTS      = CFG['paths']['results']
FIGURES      = CFG['paths']['figures']
SUPERCLASSES = CFG['data']['superclasses']
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(FIGURES, exist_ok=True)

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('GPU: not available -- running on CPU')

In [ ]:
# Map experiment name -> (loader lambda, hidden_dim)
EXPERIMENT_LOADERS = {
    'hubert_ecg_blocks8':   (lambda: load_hubert_blocks(n=8),                 CFG['model']['hubert_hidden_dim']),
    'hubert_ecg_dora_r8':   (lambda: load_hubert_peft(rank=8, use_dora=True), CFG['model']['hubert_hidden_dim']),
    'hubert_ecg_lora_r8':   (lambda: load_hubert_peft(rank=8, use_dora=False),CFG['model']['hubert_hidden_dim']),
    'leadwise_transformer': (lambda: load_leadwise(),                          CFG['model']['leadwise']['d_model']),
}

def _best_auc(exp_name):
    p = os.path.join(RESULTS, exp_name, 'history.json')
    if not os.path.exists(p):
        return 0.0
    return json.load(open(p))['best_auc']

BEST_NAME = max(EXPERIMENT_LOADERS, key=_best_auc)
BASE_AUC  = _best_auc(BEST_NAME)
print(f'Best base model : {BEST_NAME}')
print(f'Base AUC        : {BASE_AUC:.4f}')
print()
print('Loading checkpoint...')
loader_fn, hidden_dim = EXPERIMENT_LOADERS[BEST_NAME]
best_model, _         = loader_fn()
print(f'Loaded.  hidden_dim = {hidden_dim}')

In [ ]:
uncertainty_model = AleatoricWrapper(
    base_model = best_model,
    hidden_dim = hidden_dim,
)

trainable = sum(p.numel() for p in uncertainty_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in uncertainty_model.parameters())
print('Uncertainty model parameters:')
print(f'  Trainable : {trainable:,}')
print(f'  Total     : {total:,}')
print()

dummy = torch.randn(2, 12, 1000)
with torch.no_grad():
    mean_out, log_var_out = uncertainty_model(dummy)
assert mean_out.shape    == (2, 5), f'mean shape wrong: {mean_out.shape}'
assert log_var_out.shape == (2, 5), f'log_var shape wrong: {log_var_out.shape}'
print('Forward pass OK')
print(f'  mean    shape : {tuple(mean_out.shape)}')
print(f'  log_var shape : {tuple(log_var_out.shape)}')

In [ ]:
Y        = load_all_labels(
    DATA_PATH + 'ptbxl_database.csv',
    DATA_PATH + 'scp_statements.csv',
)
train_df = Y[Y.strat_fold <  9]
val_df   = Y[Y.strat_fold == 9]
test_df  = Y[Y.strat_fold == 10]

train_ds = ECGDatasetFull(train_df, DATA_PATH)
val_ds   = ECGDatasetFull(val_df,   DATA_PATH)
test_ds  = ECGDatasetFull(test_df,  DATA_PATH)

print(f'Train : {len(train_df):,} records')
print(f'Val   : {len(val_df):,} records')
print(f'Test  : {len(test_df):,} records')
print()

auc, hist, prof = run_uncertainty_experiment(
    uncertainty_model,
    train_ds,
    val_ds,
    experiment_name = 'uncertainty_model',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_peft'],
    batch_size      = CFG['training']['batch_size_full'],
    save_dir        = RESULTS,
)
print(f'Best AUC with uncertainty head : {auc:.4f}')

In [ ]:
delta    = auc - BASE_AUC
sign_str = '+' if delta >= 0 else ''

print('=' * 52)
print('AUC COMPARISON')
print('=' * 52)
print(f'  Base model ({BEST_NAME})')
print(f'    AUC : {BASE_AUC:.4f}')
print(f'  + Uncertainty head')
print(f'    AUC : {auc:.4f}')
print(f'  Difference : {sign_str}{delta:.4f}')
print('=' * 52)
print()
if abs(delta) < 0.01:
    print('Delta < 0.01 -- uncertainty head is essentially free.')
elif delta >= 0:
    print('Uncertainty head improved AUC slightly.')
else:
    print('Uncertainty head reduced AUC -- expected trade-off for calibration.')

In [ ]:
pipeline = ECGInferencePipeline(uncertainty_model, has_uncertainty=True)

N = min(500, len(test_ds))
print(f'Running inference on {N} test records...')

records = []
for i in range(N):
    x, y_true = test_ds[i]
    result    = pipeline.predict(x.numpy())

    true_classes = [
        SUPERCLASSES[j]
        for j, v in enumerate(y_true)
        if v == 1
    ]
    correct = any(c in result['predicted_classes'] for c in true_classes)
    records.append({
        'uncertainty':  result['uncertainty'],
        'confidence':   result['confidence_score'],
        'correct':      correct,
        'true_classes': true_classes,
        'pred_classes': result['predicted_classes'],
    })

df_results = pd.DataFrame(records)

median_unc   = df_results['uncertainty'].median()
low_unc      = df_results[df_results['uncertainty'] <= median_unc]
high_unc     = df_results[df_results['uncertainty'] >  median_unc]
low_unc_acc  = low_unc['correct'].mean()
high_unc_acc = high_unc['correct'].mean()

print()
print(f'Median uncertainty                       : {median_unc:.4f}')
print(f'Accuracy on LOW  uncertainty samples     : {low_unc_acc:.3f}  (n={len(low_unc)})')
print(f'Accuracy on HIGH uncertainty samples     : {high_unc_acc:.3f}  (n={len(high_unc)})')
print(f'Overall accuracy (partial-match)         : {df_results["correct"].mean():.3f}')
print()
if low_unc_acc > high_unc_acc:
    print('Uncertainty is meaningful -- less certain -> less accurate')
else:
    print('Uncertainty did not correlate with accuracy on this split.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: scatter uncertainty vs confidence, coloured by correctness
colors = df_results['correct'].map({True: 'green', False: 'red'})
axes[0].scatter(
    df_results['uncertainty'],
    df_results['confidence'],
    c=colors, alpha=0.4, s=20,
)
axes[0].axvline(median_unc, linestyle='--', color='gray', alpha=0.6, label='median unc')
axes[0].set_xlabel('Uncertainty score', fontsize=11)
axes[0].set_ylabel('Confidence score',  fontsize=11)
axes[0].set_title('Uncertainty vs Confidence', fontsize=12)
axes[0].legend(handles=[
    mpatches.Patch(color='green', label='Correct'),
    mpatches.Patch(color='red',   label='Wrong'),
], fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right: box plot of uncertainty split by prediction outcome
correct_unc = df_results[df_results['correct'] == True]['uncertainty']
wrong_unc   = df_results[df_results['correct'] == False]['uncertainty']
bp = axes[1].boxplot(
    [correct_unc, wrong_unc],
    labels=['Correct', 'Wrong'],
    patch_artist=True,
)
bp['boxes'][0].set_facecolor('lightgreen')
bp['boxes'][1].set_facecolor('lightcoral')
axes[1].set_ylabel('Uncertainty score', fontsize=11)
axes[1].set_title('Uncertainty by Prediction Outcome', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = FIGURES + 'uncertainty_analysis.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {fig_path}')

In [ ]:
def _show_example(idx, case_label):
    row    = df_results.loc[idx]
    x, _   = test_ds[idx]
    result = pipeline.predict(x.numpy())

    probs_str = '  '.join(
        f"{c}:{result['class_probabilities'][c]:.2f}"
        for c in SUPERCLASSES
    )
    true_str = ', '.join(row['true_classes']) if row['true_classes'] else 'None'
    pred_str = ', '.join(result['predicted_classes'])

    print(f'[ {case_label} ]')
    print(f'  True label      : {true_str}')
    print(f'  Predicted class : {pred_str}')
    print(f'  Class probs     : {probs_str}')
    print(f'  Confidence      : {result["confidence_score"]:.3f}')
    print(f'  Uncertainty     : {result["uncertainty"]:.4f}  ({result["uncertainty_level"]})')
    print(f'  Correct         : {"Yes" if row["correct"] else "No"}')
    print()

_corr  = df_results[df_results['correct'] == True]
_wrong = df_results[df_results['correct'] == False]

# Case 1: correct + low uncertainty (ideal)
idx1 = _corr.nsmallest(1, 'uncertainty').index[0]
# Case 2: correct + high uncertainty (model right but unsure)
idx2 = _corr.nlargest(1, 'uncertainty').index[0]
# Case 3: wrong + high uncertainty (model correctly unsure)
idx3 = (_wrong.nlargest(1, 'uncertainty').index[0]
        if len(_wrong) > 0 else df_results.index[0])
# Case 4: wrong + low uncertainty (dangerous -- overconfident)
idx4 = (_wrong.nsmallest(1, 'uncertainty').index[0]
        if len(_wrong) > 0 else df_results.index[1])
# Case 5: multi-label record
_multi = df_results[df_results['true_classes'].apply(len) > 1]
idx5   = _multi.index[0] if len(_multi) > 0 else df_results.index[2]

_show_example(idx1, 'Case 1 -- Correct + Low Uncertainty  (ideal)')
_show_example(idx2, 'Case 2 -- Correct + High Uncertainty (model right but unsure)')
_show_example(idx3, 'Case 3 -- Wrong   + High Uncertainty (model correctly unsure)')
_show_example(idx4, 'Case 4 -- Wrong   + Low Uncertainty  (dangerous -- overconfident)')
_show_example(idx5, 'Case 5 -- Multi-label case')

In [ ]:
summary = {
    'base_model_name':        BEST_NAME,
    'base_model_auc':         float(BASE_AUC),
    'uncertainty_model_auc':  float(auc),
    'auc_difference':         float(auc - BASE_AUC),
    'low_unc_accuracy':       float(low_unc_acc),
    'high_unc_accuracy':      float(high_unc_acc),
    'median_uncertainty':     float(median_unc),
    'uncertainty_meaningful': bool(low_unc_acc > high_unc_acc),
    'n_test_records':         int(N),
    'overall_accuracy':       float(df_results['correct'].mean()),
}

out_dir  = os.path.join(RESULTS, 'uncertainty_model')
out_path = os.path.join(out_dir, 'uncertainty_analysis.json')
os.makedirs(out_dir, exist_ok=True)

with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved -> {out_path}')
print()
print(json.dumps(summary, indent=2))